In [0]:
bronze_df = spark.table(
    "finance_ml.bronze.account_events_raw"
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


silver_df = (
    bronze_df

    # Converting structurally invalid values to NULL.
    .withColumn(
        "cash_outflow",
        F.when(
            F.col("cash_outflow") < 0,
            F.lit(None),
        ).otherwise(
            F.col("cash_outflow")
        ),
    )

    .withColumn(
        "credit_utilization",
        F.when(
            F.col("credit_utilization") > 1,
            F.lit(None),
        ).otherwise(
            F.col("credit_utilization")
        ),
    )
)

In [0]:
silver_df.filter(
    F.col("cash_outflow") < 0
).count()

In [0]:
silver_df.filter(
    F.col("credit_utilization") > 1
).count()

In [0]:
dedup_window = (
    Window
    .partitionBy(
        "account_id",
        "date",
    )
    .orderBy(
        *[
            F.col(column).asc_nulls_last()
            for column in bronze_df.columns
        ]
    )
)

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window),
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

In [0]:
duplicate_count = (
    silver_df
    .groupBy(
        "account_id",
        "date",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(duplicate_count)

In [0]:
print(silver_df.count())

In [0]:
account_window = (
    Window
    .partitionBy("account_id")
)

In [0]:
## Imputing missing account_type's:

# existing account_type → keep it
# otherwise → use another known account_type from same account
# otherwise → "unknown"

silver_df = (
    silver_df
    .withColumn(
        "account_type",
        F.coalesce(
            F.col("account_type"),
            F.first(
                "account_type",
                ignorenulls=True,
            ).over(account_window),
            F.lit("unknown"),
        ),
    )
)

In [0]:
print(
    silver_df
    .filter(
        F.col("account_type").isNull()
    )
    .count()
)

In [0]:
silver_df.groupBy(
    "account_type"
).count().show()

In [0]:
# Causal forward-fill imputation for the state-like numeric columns: 
# use the last originally observed past value, not an already-imputed value.
# `-1` is important: it means “look only at rows strictly before the current row.”
time_window = (
    Window
    .partitionBy("account_id")
    .orderBy("date")
    .rowsBetween(
        Window.unboundedPreceding,
        -1,
    )
)

In [0]:
state_columns = [
    "balance",
    "credit_utilization",
    "payment_ratio",
    "days_past_due",
]

In [0]:
# Then, for each state-like column, create an imputed version from the original column:
for column in state_columns:
    previous_observed = F.last(
        F.col(column),
        ignorenulls=True,
    ).over(time_window)

    silver_df = silver_df.withColumn(
        column,
        F.coalesce(
            F.col(column),
            previous_observed,
        ),
    )

In [0]:
# Now check remaining null counts:
for column in state_columns:
    missing = (
        silver_df
        .filter(
            F.col(column).isNull()
        )
        .count()
    )

    print(column, missing)

In [0]:
# Check the flow/event-like columns we intentionally decided not to forward-fill:

flow_columns = [
    "cash_inflow",
    "cash_outflow",
    "transaction_count",
    "amount_due",
    "amount_paid",
]

for column in flow_columns:
    missing = (
        silver_df
        .filter(
            F.col(column).isNull()
        )
        .count()
    )

    print(column, missing)

In [0]:
# Validate the target labels
target_columns = [
    "risk_state",
    "risk_state_name",
]

for column in target_columns:
    missing = (
        silver_df
        .filter(
            F.col(column).isNull()
        )
        .count()
    )

    print(column, missing)

In [0]:
# Now persist the cleaned DataFrame as:  finance_ml.silver.account_events_clean

(
    silver_df.write
    .mode("overwrite")
    .saveAsTable(
        "finance_ml.silver.account_events_clean"
    )
)

In [0]:
spark.sql(
    """
    SELECT COUNT(*) AS n_rows
    FROM finance_ml.silver.account_events_clean
    """
).show()

In [0]:
# Validate Silver quality
spark.sql(
    """
    SELECT
        COUNT(*) AS n_rows,

        SUM(
            CASE
                WHEN cash_outflow < 0 THEN 1
                ELSE 0
            END
        ) AS negative_cash_outflow,

        SUM(
            CASE
                WHEN credit_utilization > 1 THEN 1
                ELSE 0
            END
        ) AS invalid_credit_utilization,

        SUM(
            CASE
                WHEN risk_state IS NULL
                  OR risk_state_name IS NULL
                THEN 1
                ELSE 0
            END
        ) AS missing_targets

    FROM finance_ml.silver.account_events_clean
    """
).show()


In [0]:
spark.sql(
    """
    SELECT COUNT(*) AS duplicate_keys
    FROM (
        SELECT
            account_id,
            date,
            COUNT(*) AS n_records
        FROM finance_ml.silver.account_events_clean
        GROUP BY
            account_id,
            date
        HAVING COUNT(*) > 1
    )
    """
).show()

In [0]:
# All above in one function (improved version)

def build_silver_dataframe(bronze_df):
    """
    Cleaning Bronze account-event data into a Silver Spark DataFrame.

    Args:
        bronze_df:
            Input Spark DataFrame containing Bronze account-event data.

    Returns:
        Cleaned Spark DataFrame suitable for the Silver layer.
    """
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    silver_df = (
        bronze_df
        # Converting structurally invalid values to NULL.
        .withColumn(
            "cash_outflow",
            F.when(
                F.col("cash_outflow") < 0,
                F.lit(None),
            ).otherwise(F.col("cash_outflow")),
        ).withColumn(
            "credit_utilization",
            F.when(
                F.col("credit_utilization") > 1,
                F.lit(None),
            ).otherwise(F.col("credit_utilization")),
        )
    )

    # Removing duplicate account-date rows deterministically.
    dedup_window = (
        Window
        .partitionBy(
            "account_id",
            "date",
        )
        .orderBy(
            (
                F.col("risk_state").isNull().cast("int")
                + F.col("risk_state_name").isNull().cast("int")
            ).asc(),
            *[
                F.col(column).asc_nulls_last()
                for column in bronze_df.columns
            ],
        )
    )

    silver_df = (
        silver_df.withColumn(
            "_row_number",
            F.row_number().over(dedup_window),
        )
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    # Filling the static account type from another known row
    # belonging to the same account.
    account_window = Window.partitionBy("account_id")

    silver_df = silver_df.withColumn(
        "account_type",
        F.coalesce(
            F.col("account_type"),
            F.first(
                "account_type",
                ignorenulls=True,
            ).over(account_window),
            F.lit("unknown"),
        ),
    )

    # Using only earlier observations from the same account.
    time_window = (
        Window.partitionBy("account_id")
        .orderBy("date")
        .rowsBetween(
            Window.unboundedPreceding,
            -1,
        )
    )

    state_columns = [
        "balance",
        "credit_utilization",
        "payment_ratio",
        "days_past_due",
    ]

    for column in state_columns:
        previous_observed = F.last(
            F.col(column),
            ignorenulls=True,
        ).over(time_window)

        silver_df = silver_df.withColumn(
            column,
            F.coalesce(
                F.col(column),
                previous_observed,
            ),
        )

    # Removing rows with missing target labels.
    silver_df = silver_df.filter(
        F.col("risk_state").isNotNull() & F.col("risk_state_name").isNotNull()
    )

    return silver_df


In [0]:
from datetime import datetime

test_rows = [
    (
        "ACC_TEST",
        datetime(2026, 1, 1),
        "checking",
        None,
        100.0,
        0.4,
        0.8,
        0,
        0,
        "normal",
    ),
    (
        "ACC_TEST",
        datetime(2026, 1, 1),
        "checking",
        5000.0,
        100.0,
        0.4,
        0.8,
        0,
        None,
        None,
    ),
]

test_columns = [
    "account_id",
    "date",
    "account_type",
    "balance",
    "cash_outflow",
    "credit_utilization",
    "payment_ratio",
    "days_past_due",
    "risk_state",
    "risk_state_name",
]

test_bronze_df = spark.createDataFrame(
    test_rows,
    test_columns,
)

In [0]:
test_silver_df = build_silver_dataframe(
    test_bronze_df
)

test_silver_df.show(
    truncate=False
)